# 04b — Regression Matrix Plot

Generates a 3×N matrix of regression plots comparing multiple training runs
side by side (one column per run, one row per parameter).

Run this notebook after `04_training.ipynb`.

**Output:** `graphs_matrix/<timestamp>_<SURVEY>_<MODEL_TYPE>.png`

## Configuration

In [ ]:
import os
import json
import pandas as pd
import minas as mg
from minas.evaluation._graphics import plot_regression_matrix, show

# ── Survey and model ──────────────────────────────────────────────────────────
SURVEY     = 'SPLUS'
MODEL_TYPE = 'XGB'
PARAM_LIST = ['teff', 'logg', 'feh']
PREDICTS_DIR = 'predicts'

# ── Training runs to compare ──────────────────────────────────────────────────
# Add one entry per column in the matrix.
# 'title'      : label shown at the top of the column
# 'training_id': datetime string from 04_training.ipynb
RUNS = [
    {'title': 'Run A', 'training_id': '20251027174628'},
    {'title': 'Run B', 'training_id': '20251027175631'},
    # add more runs here if needed
]

# ── Evaluation bins ───────────────────────────────────────────────────────────
BINS = {
    'teff': [3000, 4000, 5000, 6000, 7000, 8000],
    'logg': [1.0, 2.0, 3.0, 4.0, 5.0],
    'feh' : [-2.5, -1.5, -0.5, 0.5, 1.0],
}

# ── Plot settings ─────────────────────────────────────────────────────────────
POINT_SIZE = 4
COLOR_MODE = 'cmap'   # 'cmap' (coloured by bin) or 'color' (solid colour per parameter)

datetime_str = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
print(f'Runs to compare: {[r["title"] for r in RUNS]}')

## Load prediction results

In [ ]:
results_dict      = {}
metrics_json_paths = {}
titles            = [r['title'] for r in RUNS]

for run in RUNS:
    title = run['title']
    tid   = run['training_id']

    for param in PARAM_LIST:
        key          = f'{param}_{title}'
        results_path = os.path.join(PREDICTS_DIR, f'{tid}_{SURVEY}_{param}_{MODEL_TYPE}.csv')
        metrics_path = results_path.replace('.csv', '_metrics.json')

        if not os.path.exists(results_path):
            print(f'  [missing] {results_path}')
            results_dict[key]       = None
            metrics_json_paths[key] = None
            continue

        df = pd.read_csv(results_path, index_col=0)
        results_dict[key]       = (df['true'], df['predicted'])
        metrics_json_paths[key] = metrics_path if os.path.exists(metrics_path) else None
        print(f'  [ok] {param} / {title}  ({len(df):,} objects)')

print('\nLoading complete.')

## Generate regression matrix

In [ ]:
fig = plot_regression_matrix(
    results_dict=results_dict,
    bins_dict=BINS,
    param_order=PARAM_LIST,
    titles=titles,
    point_size=POINT_SIZE,
    metrics_json_paths=metrics_json_paths,
    color_mode=COLOR_MODE,
)

graphs_dir = 'graphs_matrix'
os.makedirs(graphs_dir, exist_ok=True)
out_path = os.path.join(graphs_dir, f'{datetime_str}_{SURVEY}_{MODEL_TYPE}.png')
fig.savefig(out_path, bbox_inches='tight', dpi=150)
print(f'Figure saved: {out_path}')
show(fig)